# Caso 04: VueltaFit — Gimnasios: ¿Qué Plan Deja Más Dinero?
## Día 1: Bienvenida al Negocio y Primer Contacto con los Datos (EDA)

**Objetivo de hoy:**
1. **Comprender el modelo de negocio:**
   - La economía de la suscripción con asistencia en VueltaFit (8 sedes, ~9.000 socios activos).
   - **Tensión clave:** Rubén (Growth) quiere empujar la campaña *"Pásate al Anual"* por su elevado LTV de prepago (~460 € vs ~300 €). Nuria (Datos) y Marta (CEO) saben que la rentabilidad no está en el grifo (altas) sino en tapar el desagüe del *churn* (renovación).
   - **Regla de oro:** Pagar $\neq$ venir $\neq$ quedarse. Retener cuesta entre 5 y 7 veces menos que captar.
2. **Auditoría Forense de las 7 exportaciones:**
   - Formular a cada fichero las 4 preguntas de rigor: `.shape`, `.head()`, `.dtypes` y `.isnull().sum()`.
   - Identificar las trampas advertidas por Álex (duplicados de check-in en el mismo minuto, mes en inglés, fecha de baja vacía = activo).


In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
RAW_DIR = os.path.join('data', 'raw')


### 1. Auditoría: `socios.csv`
**Rol en el modelo:** Tabla de dimensión central con los `9.000` socios de la cadena, su sede asignada, canal de captación, edad, sexo y fechas de alta/baja.

In [2]:
socios = pd.read_csv(os.path.join(RAW_DIR, 'socios.csv'))
print(f"=== socios.csv ===")
print("Dimensiones (filas, columnas):", socios.shape)
print("\nTipos detectados:\n", socios.dtypes)
nulls = socios.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
socios.head(3)

=== socios.csv ===
Dimensiones (filas, columnas): (9000, 7)

Tipos detectados:
 socio_id           int64
sede_id            int64
fecha_alta           str
canal_captacion      str
edad               int64
sexo                 str
fecha_baja           str
dtype: object

Valores nulos:
 fecha_baja    4370
dtype: int64


,socio_id,sede_id,fecha_alta,canal_captacion,edad,sexo,fecha_baja
0,1,7,26/06/2023,walk-in,46,F,2024-06-25
1,2,1,2023-09-18,Walk-in,29,F,17/01/2024
2,3,7,2023-09-01,promo,34,hombre,NaN


**Hallazgos en `socios.csv`:**
* **Dimensiones:** `9.000` filas y `7` columnas (`socio_id` único).
* **Formatos mixtos y texto sucio:** `fecha_alta` y `fecha_baja` mezclan formato europeo (`26/06/2023`) e ISO (`2023-09-18`); `canal_captacion` y `sexo` presentan variantes de mayúsculas/minúsculas (`walk-in`, `Walk-in`, `F`, `hombre`).
* **Vacío con significado:** Los **`4.370` nulos en `fecha_baja`** no son datos perdidos, sino los **socios que continúan en activo** a fecha de corte (`30/06/2024`).

---
### 2. Auditoría: `suscripciones.csv`
**Rol en el modelo:** Tabla de contratos (`10.957` filas): recoge cada alta y renovación, el tipo de plan contratado, fechas de inicio/fin/baja, precio y estado.

In [3]:
suscripciones = pd.read_csv(os.path.join(RAW_DIR, 'suscripciones.csv'))
print(f"=== suscripciones.csv ===")
print("Dimensiones (filas, columnas):", suscripciones.shape)
print("\nTipos detectados:\n", suscripciones.dtypes)
nulls = suscripciones.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
suscripciones.head(3)

=== suscripciones.csv ===
Dimensiones (filas, columnas): (10957, 9)

Tipos detectados:
 suscripcion_id    int64
socio_id          int64
tipo_plan           str
fecha_inicio        str
fecha_fin           str
fecha_baja          str
precio              str
estado              str
motivo_baja         str
dtype: object

Valores nulos:
 fecha_baja     7322
motivo_baja    7322
dtype: int64


,suscripcion_id,socio_id,tipo_plan,fecha_inicio,fecha_fin,fecha_baja,precio,estado,motivo_baja
0,500001,1,ANUAL,26/06/2023,25 Jun 2024,NaN,"449,00 €",vencida,NaN
1,500002,2,mensual,2023-09-18,17/09/2024,2024-01-17,"39,00 €",baja,sin uso
2,500003,3,anual,2023-09-01,31/08/2024,NaN,"420,00 €",activa,NaN


**Hallazgos en `suscripciones.csv`:**
* **Dimensiones:** `10.957` contratos para `9.000` socios (existen socios con más de una suscripción encadenada, lo que nos permitirá medir la **renovación real**).
* **Trampas detectadas:** `tipo_plan` aparece con mayúsculas/minúsculas y sinónimos (`ANUAL`, `anual`, `mensual`); `fecha_fin` incluye meses en inglés (`25 Jun 2024`); `precio` es texto con coma decimal y símbolo `€` (`449,00 €`).
* **Vacíos con significado:** `7.322` nulos en `fecha_baja` y `motivo_baja` corresponden a contratos que no sufrieron cancelación anticipada (siguen activos o llegaron a su vencimiento natural).

---
### 3. Auditoría: `checkins.csv`
**Rol en el modelo:** Tabla de hechos de asistencia física por torniquete (`236.097` registros).

In [4]:
checkins = pd.read_csv(os.path.join(RAW_DIR, 'checkins.csv'))
print(f"=== checkins.csv ===")
print("Dimensiones (filas, columnas):", checkins.shape)
print("\nTipos detectados:\n", checkins.dtypes)
nulls = checkins.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
checkins.head(3)

=== checkins.csv ===
Dimensiones (filas, columnas): (236097, 5)

Tipos detectados:
 checkin_id     int64
socio_id       int64
sede_id          str
fecha_hora       str
tipo_acceso      str
dtype: object

Valores nulos:
 fecha_hora    40
dtype: int64


,checkin_id,socio_id,sede_id,fecha_hora,tipo_acceso
0,1216214,3313,madrid,11/11/2023 07:28,Torniquete
1,1235617,7553,BCN,2024-03-24 19:56:36,torniquete
2,1033608,6166,MALAGA,06/12/2023 08:37,Torniquete


**Hallazgos en `checkins.csv`:**
* **Dimensiones:** `236.097` filas y `5` columnas, con `40` nulos en `fecha_hora`.
* **Trampas detectadas:** `sede_id` no contiene el ID numérico sino nombres y siglas en texto sucio (`madrid`, `BCN`, `MALAGA`), y `fecha_hora` combina formatos con y sin segundos (`11/11/2023 07:28` vs `2024-03-24 19:56:36`).

---
### 4. Auditoría: `clases.csv`
**Rol en el modelo:** Catálogo de `5.000` sesiones de clases colectivas dirigidas ofertadas por sede, monitor, horario y aforo.

In [5]:
clases = pd.read_csv(os.path.join(RAW_DIR, 'clases.csv'))
print(f"=== clases.csv ===")
print("Dimensiones (filas, columnas):", clases.shape)
print("\nTipos detectados:\n", clases.dtypes)
nulls = clases.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
clases.head(3)

=== clases.csv ===
Dimensiones (filas, columnas): (5000, 6)

Tipos detectados:
 clase_id      int64
sede_id         str
tipo_clase      str
monitor         str
fecha_hora      str
aforo         int64
dtype: object

Valores nulos:
 Sin nulos


,clase_id,sede_id,tipo_clase,monitor,fecha_hora,aforo
0,700001,Zaragoza,HIIT,Hugo Castro,27/01/2023 07:30,19
1,700002,barcelona,HIIT,PAULA NAVARRO,22/05/2023 10:00,27
2,700003,Madrid,bodypump,María Sanz,17 Feb 2024 21:00,26


**Hallazgos en `clases.csv`:**
* **Dimensiones:** `5.000` filas y `6` columnas (`0` nulos).
* **Trampas detectadas:** Al igual que en `checkins`, `sede_id` viene como nombre de ciudad con grafías dispares (`Zaragoza`, `barcelona`, `Madrid`), `tipo_clase` y `monitor` mezclan cajas (`HIIT`, `bodypump`, `PAULA NAVARRO`) y `fecha_hora` incluye meses en inglés (`17 Feb 2024 21:00`).

---
### 5. Auditoría: `reservas.csv`
**Rol en el modelo:** Tabla de hechos de reservas de clases colectivas realizadas por los socios (`102.424` filas).

In [6]:
reservas = pd.read_csv(os.path.join(RAW_DIR, 'reservas.csv'))
print(f"=== reservas.csv ===")
print("Dimensiones (filas, columnas):", reservas.shape)
print("\nTipos detectados:\n", reservas.dtypes)
nulls = reservas.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
reservas.head(3)

=== reservas.csv ===
Dimensiones (filas, columnas): (102424, 6)

Tipos detectados:
 reserva_id       int64
socio_id         int64
clase_id         int64
fecha_reserva      str
fecha_clase        str
estado             str
dtype: object

Valores nulos:
 Sin nulos


,reserva_id,socio_id,clase_id,fecha_reserva,fecha_clase,estado
0,987519,7689,702142,2024-01-19,2024-01-20,asistio
1,948472,4249,701649,2024-02-21,25/02/2024,asistió
2,994637,8328,704680,2023-03-20,2023-03-25,no_show


**Hallazgos en `reservas.csv`:**
* **Dimensiones:** `102.424` filas y `6` columnas (`0` nulos).
* **Trampas detectadas:** `estado` presenta variantes con y sin tilde (`asistio`, `asistió`, `no_show`, `cancelada`) y las fechas combinan `YYYY-MM-DD` con `DD/MM/YYYY`.

---
### 6. Auditoría: `pagos.csv`
**Rol en el modelo:** Tabla de transacciones financieras (`45.637` cobros recurrentes y prepagos anuales de pasarela/recibo).

In [7]:
pagos = pd.read_csv(os.path.join(RAW_DIR, 'pagos.csv'))
print(f"=== pagos.csv ===")
print("Dimensiones (filas, columnas):", pagos.shape)
print("\nTipos detectados:\n", pagos.dtypes)
nulls = pagos.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
pagos.head(3)

=== pagos.csv ===
Dimensiones (filas, columnas): (45637, 7)

Tipos detectados:
 pago_id           int64
socio_id          int64
suscripcion_id    int64
fecha_pago          str
importe             str
metodo              str
estado              str
dtype: object

Valores nulos:
 Sin nulos


,pago_id,socio_id,suscripcion_id,fecha_pago,importe,metodo,estado
0,325273,4856,505942,24/03/2024,"34,90 €",Domiciliacion,ok
1,339562,7760,509453,24/01/2024,"39,00 €",recibo,ok
2,335404,6929,508444,27 Apr 2024,"420,00 €",tarjeta,ok


**Hallazgos en `pagos.csv`:**
* **Dimensiones:** `45.637` filas y `7` columnas (`0` nulos).
* **Trampas detectadas:** `importe` viene formateado en texto con coma y `€` (`34,90 €`, `420,00 €`), `fecha_pago` mezcla formatos con inglés (`27 Apr 2024`), y `metodo`/`estado` requieren normalización y filtrado de reembolsos e importes anómalos antes de sumar el LTV.

---
### 7. Auditoría: `sedes.csv`
**Rol en el modelo:** Tabla maestra de los `8` gimnasios de VueltaFit en `6` ciudades, con su aforo y cuota base.

In [8]:
sedes = pd.read_csv(os.path.join(RAW_DIR, 'sedes.csv'))
print(f"=== sedes.csv ===")
print("Dimensiones (filas, columnas):", sedes.shape)
print("\nTipos detectados:\n", sedes.dtypes)
nulls = sedes.isnull().sum()
print("\nValores nulos:\n", nulls[nulls > 0] if nulls.any() else "Sin nulos")
sedes.head(3)

=== sedes.csv ===
Dimensiones (filas, columnas): (8, 6)

Tipos detectados:
 sede_id           int64
ciudad              str
nombre              str
aforo             int64
cuota_base          str
fecha_apertura      str
dtype: object

Valores nulos:
 Sin nulos


,sede_id,ciudad,nombre,aforo,cuota_base,fecha_apertura
0,1,Madrid,VueltaFit Chamberí,850,"42,00 €",01/03/2020
1,2,Madrid,VueltaFit Vallecas,700,"39,00 €",2021-09-01
2,3,Barcelona,VueltaFit Eixample,820,"44,00 €",2020-10-01


**Hallazgos en `sedes.csv`:**
* **Dimensiones:** `8` filas (`sede_id` del `1` al `8` repartidos en Madrid [2], Barcelona [2], Valencia, Sevilla, Zaragoza y Málaga) y `6` columnas (`0` nulos). `cuota_base` está en formato texto (`42,00 €`).

---
### 8. Inspección Detallada de las Trampas de Álex
Cuantificamos en crudo el impacto exacto de los avisos del equipo técnico: duplicados de torniquete en el mismo minuto, las 8 grafías de `tipo_plan`, los vacíos con significado en `fecha_baja` y los outliers de edad.

In [9]:
# 1. Check-ins: Duplicados generados por reintento en el torniquete (mismo socio en el mismo minuto)
print("Total checkins brutos:", len(checkins))
print("Checkin_id únicos:", checkins['checkin_id'].nunique(), "(los IDs son nuevos, la trampa no se ve por ID)")

fh_chk = pd.to_datetime(checkins['fecha_hora'], format='mixed', dayfirst=True, errors='coerce')
checkins['minuto'] = fh_chk.dt.floor('min')
dups_chk = checkins.duplicated(subset=['socio_id', 'minuto']).sum()
print(f"Check-ins duplicados (mismo socio y mismo minuto): {dups_chk} entradas a deduplicar mañana.")

# 2. Suscripciones: Las 8 grafías de tipo_plan
print("\nVariantes de tipo_plan en suscripciones:")
print(suscripciones['tipo_plan'].value_counts())

# 3. Socios y Suscripciones: Nulos con significado en fecha_baja
socios_activos = socios['fecha_baja'].isnull().sum()
susc_activas = suscripciones['fecha_baja'].isnull().sum()
print(f"\nSocios con fecha_baja vacía (= activos): {socios_activos} de {len(socios)}")
print(f"Suscripciones con fecha_baja vacía (= en curso): {susc_activas} de {len(suscripciones)}")

# 4. Outliers en Edad (socios)
edades_raras = socios[(socios['edad'] < 14) | (socios['edad'] > 90)]['edad'].tolist()
print(f"\nEdades imposibles detectadas: {edades_raras}")


Total checkins brutos: 236097
Checkin_id únicos: 236097 (los IDs son nuevos, la trampa no se ve por ID)


Check-ins duplicados (mismo socio y mismo minuto): 916 entradas a deduplicar mañana.

Variantes de tipo_plan en suscripciones:
tipo_plan
mensual    3641
anual      2355
Mensual    1734
Anual      1044
mes         668
MENSUAL     625
año         449
ANUAL       441
Name: count, dtype: int64

Socios con fecha_baja vacía (= activos): 4370 de 9000
Suscripciones con fecha_baja vacía (= en curso): 7322 de 10957

Edades imposibles detectadas: [150, 0, 8, 0, 150, 8, 150, 0, 8]


## Síntesis Forense: Inventario de Minas para el Día 2

Tras completar la exploración de los 7 ficheros de VueltaFit, se documentan las siguientes conclusiones forenses:
1. **Granularidad y Volumetría Asimétrica:**
   - Hay dos órdenes de magnitud entre las tablas de estado (`socios` con 9.000 filas, `suscripciones` con 10.957) y las tablas de comportamiento (`reservas` con 102.424 filas y `checkins` con 236.097 filas).
   - El comportamiento (asistencia) es el rastro continuo que explicará la retención.
2. **Trampas Técnicas:**
   - **Check-ins fantasma:** El lector duplica entradas generando un `checkin_id` nuevo. Se identifican **916 duplicados** en el mismo minuto para un mismo socio.
   - **Planes fragmentados:** `tipo_plan` viene en 8 grafías distintas (`mensual`, `anual`, `Mensual`, `Anual`, `mes`, `MENSUAL`, `año`, `ANUAL`) que deben mapearse a 2 categorías canónicas.
   - **Precios e importes como texto:** `suscripciones.precio`, `pagos.importe` y `sedes.cuota_base` vienen como texto con el símbolo `€`.
   - **Edades imposibles:** Registros con 0, 8 y 150 años que deben marcarse como outliers.
   - **Nulo con significado:** `fecha_baja` vacía en 4.370 socios y 7.322 suscripciones no es un error de volcado; es el indicador de que siguen activos.
